# Building Soccer Prediction Models with PyTorch

**Chapter 7: Predicting Soccer Outcomes with Deep Learning**

## Learning Objectives

- Set up PyTorch for deep learning
- Prepare soccer data for neural network training
- Build binary classification models (home win prediction)
- Build multi-class classification models (win/draw/loss)
- Train and evaluate neural networks
- Understand the complete workflow from data to predictions

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

# Set random seeds for reproducibility
np.random.seed(42)
torch.manual_seed(42)

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

## Introduction to PyTorch

**PyTorch** is a powerful open-source framework designed for building and training neural networks. It's known for:
- User-friendly interface
- Robust GPU acceleration
- Extensive library ecosystem
- Dynamic computation graphs

In this notebook, we'll build practical soccer predictors with increasing difficulty:
1. **Task 1 (Binary)**: Will the home team win?
2. **Task 2 (Multi-class)**: Win / Draw / Loss prediction

Let's start with the simplest task and work our way up!

## Generate Simulated Soccer Data

For this tutorial, we'll create realistic soccer match data. In practice, you would load this from a CSV file or API.

In [ ]:
# Generate simulated match data
np.random.seed(42)
n_matches = 1000

# Features: shots, possession%, xG (expected goals), corners, fouls
shots_home = np.random.poisson(12, n_matches)
possession_home = np.random.normal(50, 15, n_matches)
xg_home = np.random.gamma(2, 0.6, n_matches)
corners_home = np.random.poisson(5, n_matches)
fouls_home = np.random.poisson(10, n_matches)

shots_away = np.random.poisson(10, n_matches)
possession_away = 100 - possession_home
xg_away = np.random.gamma(1.5, 0.6, n_matches)
corners_away = np.random.poisson(4, n_matches)
fouls_away = np.random.poisson(11, n_matches)

# Create DataFrame
df = pd.DataFrame({
    'shots_home': shots_home,
    'possession_home': possession_home,
    'xg_home': xg_home,
    'corners_home': corners_home,
    'fouls_home': fouls_home,
    'shots_away': shots_away,
    'possession_away': possession_away,
    'xg_away': xg_away,
    'corners_away': corners_away,
    'fouls_away': fouls_away
})

# Generate outcomes based on features (with some randomness)
win_probability = 1 / (1 + np.exp(-(0.3 * (df['xg_home'] - df['xg_away']) + 
                                     0.02 * (df['shots_home'] - df['shots_away']) +
                                     0.01 * (df['possession_home'] - 50))))

df['home_win'] = (np.random.random(n_matches) < win_probability).astype(int)

# Create full-time result (FTR)
def generate_ftr(row):
    score_diff = row['xg_home'] - row['xg_away'] + np.random.normal(0, 0.5)
    if score_diff > 0.5:
        return 'H'  # Home win
    elif score_diff < -0.5:
        return 'A'  # Away win
    else:
        return 'D'  # Draw

df['FTR'] = df.apply(generate_ftr, axis=1)

print(f"Dataset shape: {df.shape}")
print(f"\nFirst few rows:")
print(df.head())
print(f"\nHome win rate: {df['home_win'].mean():.2%}")
print(f"\nFTR distribution:")
print(df['FTR'].value_counts(normalize=True))

---

# Task 1: Predicting Home Team Wins

Let's begin with the simplest prediction task: estimating the probability that the home team wins.

This is a **binary classification** problem. The model will look at pre-match features (shots, possession, xG, etc.) and output a number between 0 and 1 that we interpret as a win probability.

Just as a coach studies a team's past performances to estimate whether they're likely to win the next match, our model will learn from historical match data.

## Data Preparation

Before we can train a model, we need to put our data into a format PyTorch understands.

PyTorch provides a handy tool called **DataLoader**, which takes in tensors and produces mini-batches for training. A mini-batch is just a small slice of the data - rather than feeding the entire season of matches into the model at once, we present it with a handful of matches at a time.

In [ ]:
# Select features for binary classification
feature_cols = ['shots_home', 'possession_home', 'xg_home', 'corners_home', 'fouls_home',
                'shots_away', 'possession_away', 'xg_away', 'corners_away', 'fouls_away']

X = df[feature_cols].values
y = df['home_win'].values

# Standardize features (important for neural networks)
scaler = StandardScaler()
X = scaler.fit_transform(X)

print(f"Features shape: {X.shape}")
print(f"Target shape: {y.shape}")
print(f"\nFeature statistics after scaling:")
print(f"Mean: {X.mean(axis=0)}")
print(f"Std: {X.std(axis=0)}")

In [ ]:
# Convert to PyTorch tensors
X_tensor = torch.tensor(X, dtype=torch.float32)
y_tensor = torch.tensor(y, dtype=torch.float32)  # float targets for BCE

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X_tensor, y_tensor, test_size=0.2, random_state=42
)

# Create DataLoaders (mini-batches)
train_data = TensorDataset(X_train, y_train)
test_data = TensorDataset(X_test, y_test)

train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
test_loader = DataLoader(test_data, batch_size=32, shuffle=False)

print(f"Training set: {len(X_train)} matches")
print(f"Test set: {len(X_test)} matches")
print(f"\nBatch size: 32")
print(f"Number of training batches: {len(train_loader)}")
print(f"Number of test batches: {len(test_loader)}")

### What Just Happened?

1. **Converted to tensors**: PyTorch's version of arrays
2. **Split the dataset**: 80% training, 20% testing
3. **Created DataLoaders**: Will give us shuffled batches of size 32 during training

Shuffling ensures the model doesn't learn anything artificial from the order of matches. Just as you wouldn't want a team to practice only in the order of its fixtures, you want them to mix drills so they're ready for any opponent.

## Model Architecture

Now we define the neural network itself. For this task, we'll use a simple **Multi-Layer Perceptron (MLP)**:
- One hidden layer with 16 neurons
- ReLU activation
- Single output neuron for win probability

In [ ]:
class BinaryMLP(nn.Module):
    def __init__(self, input_size, hidden_size=16):
        super().__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden_size, 1)  # one output neuron
        
    def forward(self, x):
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        return x  # return logits (will apply sigmoid in loss function)

# Create model
input_size = X_train.shape[1]
model_binary = BinaryMLP(input_size, hidden_size=16)

# Loss function and optimizer
criterion_binary = nn.BCEWithLogitsLoss()  # stable binary cross-entropy
optimizer_binary = optim.Adam(model_binary.parameters(), lr=1e-3)

print(f"Model architecture:")
print(model_binary)
print(f"\nTotal parameters: {sum(p.numel() for p in model_binary.parameters())}")

### Understanding the Architecture

- **Input layer**: Matches the number of features (10 in our case)
- **Hidden layer**: 16 neurons that learn tactical patterns (e.g., "dominates possession and has a strong striker")
- **Output layer**: One neuron giving a score (logit) that will be converted to win probability

We use **BCEWithLogitsLoss** (Binary Cross-Entropy with Logits) which combines sigmoid activation and BCE loss for numerical stability.

The **Adam optimizer** is like a flexible coach who adapts strategy from game to game, making updates efficiently.

## Training the Model

Now comes the learning process. Training is an iterative cycle where the model:
1. Sees batches of matches
2. Makes predictions
3. Compares them against reality
4. Adjusts its weights accordingly

In [ ]:
# Training loop
num_epochs = 100
loss_history = []

for epoch in range(num_epochs):
    model_binary.train()
    running_loss = 0.0
    
    for inputs, targets in train_loader:
        # Zero the gradients
        optimizer_binary.zero_grad()
        
        # Forward pass
        logits = model_binary(inputs).squeeze(1)  # (batch_size,)
        
        # Compute loss
        loss = criterion_binary(logits, targets)
        
        # Backward pass
        loss.backward()
        
        # Update weights
        optimizer_binary.step()
        
        running_loss += loss.item()
    
    # Calculate average loss for this epoch
    avg_loss = running_loss / len(train_loader)
    loss_history.append(avg_loss)
    
    # Print progress every 10 epochs
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1:3d}/{num_epochs} - Loss: {avg_loss:.4f}")

print("\nTraining complete!")

### What Happened During Training?

Each **epoch** is like a season - the team plays through all training matches once.

For every mini-batch:
1. **Forward pass**: Model makes its best guess at who wins
2. **Loss calculation**: Compares guess to actual result
3. **Backpropagation**: Identifies which weights were responsible for mistakes
4. **Weight update**: Optimizer nudges weights toward better predictions

By repeating this 100 times, the model gradually learns patterns that link pre-match features to outcomes.

In [ ]:
# Visualize training progress
plt.figure(figsize=(10, 5))
plt.plot(loss_history, linewidth=2)
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Loss', fontsize=12)
plt.title('Training Loss Over Time', fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3)
plt.show()

print(f"Initial loss: {loss_history[0]:.4f}")
print(f"Final loss: {loss_history[-1]:.4f}")
print(f"Improvement: {(1 - loss_history[-1]/loss_history[0])*100:.1f}%")

## Evaluating the Model

Now let's see how well the model generalizes to unseen matches. This is where the test set comes in.

In [ ]:
# Evaluation
model_binary.eval()
with torch.no_grad():
    logits = model_binary(X_test).squeeze(1)
    probs = torch.sigmoid(logits)  # convert logits to probabilities
    preds = (probs >= 0.5).float()  # threshold at 0.5

# Calculate accuracy
acc = (preds == y_test).float().mean().item()
print(f"Test Accuracy: {acc:.3f} ({acc*100:.1f}%)")

# Detailed classification report
y_test_np = y_test.numpy()
preds_np = preds.numpy()

print("\nClassification Report:")
print(classification_report(y_test_np, preds_np, target_names=['No Win', 'Win']))

# Confusion matrix
cm = confusion_matrix(y_test_np, preds_np)
print("\nConfusion Matrix:")
print(cm)

In [ ]:
# Visualize confusion matrix
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['No Win', 'Win'],
            yticklabels=['No Win', 'Win'])
plt.xlabel('Predicted', fontsize=12)
plt.ylabel('Actual', fontsize=12)
plt.title('Confusion Matrix: Home Win Prediction', fontsize=14, fontweight='bold')
plt.show()

### Understanding the Results

We switch the model to **evaluation mode**, which disables training-specific features like dropout.

The raw outputs are **logits**, so we apply a **sigmoid function** to convert them to probabilities. A value of 0.73 means "the model believes there's a 73% chance the home team wins this match."

We then apply a threshold (0.5) to turn probabilities into binary predictions.

The **accuracy** tells us what fraction of test matches were predicted correctly. This is like asking: out of all the games we didn't let the model see beforehand, how many did it call right?

In [ ]:
# Look at some example predictions
print("Sample predictions:")
print("\nActual | Predicted Prob | Prediction")
print("-" * 40)
for i in range(10):
    actual = int(y_test[i].item())
    prob = probs[i].item()
    pred = int(preds[i].item())
    print(f"{actual:^6} | {prob:^14.3f} | {pred:^10}")

---

# Task 2: Predicting Game Results (Win/Draw/Loss)

Now let's tackle a more realistic task: predicting the **full-time result (FTR)** of a soccer match, which can be:
- **H**: Home win
- **D**: Draw  
- **A**: Away win

Unlike Task 1 where we only had to decide "win or not," we now face three possible outcomes. This is called a **multi-class classification** problem.

## Data Preparation for Multi-Class

We prepare the data similarly, but the target variable is now categorical. We'll use a **LabelEncoder** to convert H/D/A into numeric class labels 0, 1, and 2.

In [ ]:
# Convert categorical FTR labels to numeric
le = LabelEncoder()
y_multiclass = le.fit_transform(df['FTR'])

print(f"Label encoding:")
for i, label in enumerate(le.classes_):
    print(f"  {label} -> {i}")

print(f"\nClass distribution:")
for i, label in enumerate(le.classes_):
    count = (y_multiclass == i).sum()
    print(f"  {label}: {count} ({count/len(y_multiclass)*100:.1f}%)")

In [ ]:
# Convert to tensors
y_multiclass_tensor = torch.tensor(y_multiclass, dtype=torch.long)  # long type for CrossEntropyLoss

# Train-test split
X_train_mc, X_test_mc, y_train_mc, y_test_mc = train_test_split(
    X_tensor, y_multiclass_tensor, test_size=0.2, random_state=42
)

# Create DataLoaders
train_data_mc = TensorDataset(X_train_mc, y_train_mc)
test_data_mc = TensorDataset(X_test_mc, y_test_mc)

train_loader_mc = DataLoader(train_data_mc, batch_size=32, shuffle=True)
test_loader_mc = DataLoader(test_data_mc, batch_size=32, shuffle=False)

print(f"Training set: {len(X_train_mc)} matches")
print(f"Test set: {len(X_test_mc)} matches")

## Multi-Class Model Architecture

We now define a network that can output **three scores** instead of one. Each score corresponds to one possible result: home win, draw, or away win.

In [ ]:
class MultiClassMLP(nn.Module):
    def __init__(self, input_size, hidden_size=16, num_classes=3):
        super().__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden_size, num_classes)  # three outputs
        
    def forward(self, x):
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)  # logits for each class
        return x

# Create model
model_multiclass = MultiClassMLP(input_size, hidden_size=16, num_classes=3)

# Loss function and optimizer
criterion_multiclass = nn.CrossEntropyLoss()  # for multi-class problems
optimizer_multiclass = optim.Adam(model_multiclass.parameters(), lr=1e-3)

print(f"Model architecture:")
print(model_multiclass)
print(f"\nTotal parameters: {sum(p.numel() for p in model_multiclass.parameters())}")

### Key Difference from Binary Classification

The output layer now has **three neurons**, one for each class. The model outputs raw scores called **logits**.

**CrossEntropyLoss** automatically applies softmax to convert these logits into probabilities that sum to one. In soccer terms, the model is learning to distribute its confidence between the three possible outcomes.

## Training the Multi-Class Model

In [ ]:
# Training loop
num_epochs = 100
loss_history_mc = []

for epoch in range(num_epochs):
    model_multiclass.train()
    running_loss = 0.0
    
    for inputs, labels in train_loader_mc:
        optimizer_multiclass.zero_grad()
        
        # Forward pass
        outputs = model_multiclass(inputs)  # logits for 3 classes
        
        # Compute loss
        loss = criterion_multiclass(outputs, labels)
        
        # Backward pass
        loss.backward()
        
        # Update weights
        optimizer_multiclass.step()
        
        running_loss += loss.item()
    
    avg_loss = running_loss / len(train_loader_mc)
    loss_history_mc.append(avg_loss)
    
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1:3d}/{num_epochs} - Loss: {avg_loss:.4f}")

print("\nTraining complete!")

In [ ]:
# Visualize training progress
plt.figure(figsize=(10, 5))
plt.plot(loss_history_mc, linewidth=2, color='green')
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Loss', fontsize=12)
plt.title('Multi-Class Training Loss Over Time', fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3)
plt.show()

## Evaluating the Multi-Class Model

Evaluation looks similar, except now we're dealing with three classes. We'll measure accuracy and look at the confusion matrix to see which results are most often misclassified.

In [ ]:
# Evaluation
model_multiclass.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for inputs, labels in test_loader_mc:
        outputs = model_multiclass(inputs)
        _, predicted = torch.max(outputs, 1)  # pick class with highest logit
        all_preds.extend(predicted.numpy())
        all_labels.extend(labels.numpy())

all_preds = np.array(all_preds)
all_labels = np.array(all_labels)

# Calculate accuracy
acc_mc = accuracy_score(all_labels, all_preds)
print(f"Test Accuracy: {acc_mc:.3f} ({acc_mc*100:.1f}%)")

# Classification report
print("\nClassification Report:")
print(classification_report(all_labels, all_preds, target_names=le.classes_))

# Confusion matrix
cm_mc = confusion_matrix(all_labels, all_preds)
print("\nConfusion Matrix:")
print(cm_mc)

In [ ]:
# Visualize confusion matrix
plt.figure(figsize=(8, 6))
sns.heatmap(cm_mc, annot=True, fmt='d', cmap='Greens',
            xticklabels=le.classes_,
            yticklabels=le.classes_)
plt.xlabel('Predicted', fontsize=12)
plt.ylabel('Actual', fontsize=12)
plt.title('Confusion Matrix: Match Result Prediction', fontsize=14, fontweight='bold')
plt.show()

### Interpreting Multi-Class Results

The **confusion matrix** is particularly useful here. It shows:
- **Diagonal**: Correct predictions
- **Off-diagonal**: Misclassifications

For example, you might notice that draws are harder to predict than wins/losses. This is common in soccer - draws often depend on subtle factors that are hard to capture in statistics alone.

In [ ]:
# Look at prediction probabilities
model_multiclass.eval()
with torch.no_grad():
    sample_outputs = model_multiclass(X_test_mc[:10])
    sample_probs = torch.softmax(sample_outputs, dim=1)

print("Sample predictions with probabilities:")
print("\nActual | Predicted | P(H)  | P(D)  | P(A)")
print("-" * 50)
for i in range(10):
    actual = le.classes_[y_test_mc[i]]
    pred = le.classes_[all_preds[i]]
    ph, pd, pa = sample_probs[i].numpy()
    print(f"{actual:^6} | {pred:^9} | {ph:.3f} | {pd:.3f} | {pa:.3f}")

## Summary

In this notebook, you learned:

✅ How to prepare soccer data for PyTorch neural networks

✅ How to build and train a **binary classification** model (home win prediction)

✅ How to build and train a **multi-class classification** model (win/draw/loss)

✅ The complete workflow: data → model → training → evaluation

✅ How to interpret model predictions and probabilities

✅ The importance of proper data splitting and evaluation

### Next Steps

In the next notebook, we'll explore **regression tasks** - predicting the actual number of goals scored, and build **multi-output models** that predict both home and away goals simultaneously!

## Practice Exercises

1. **Add more features**: Include additional match statistics (yellow cards, offsides, etc.) and see if accuracy improves.

2. **Experiment with architecture**: Try different numbers of hidden neurons (8, 32, 64) or add a second hidden layer.

3. **Tune hyperparameters**: Adjust the learning rate (try 0.01, 0.001, 0.0001) and batch size (16, 64, 128).

4. **Class imbalance**: If one class dominates, try using weighted loss functions or oversampling techniques.

5. **Probability calibration**: Instead of using 0.5 as the threshold, find the optimal threshold that maximizes accuracy or F1-score.

6. **Feature importance**: Which features are most important? Try training models with different feature subsets.

7. **Early stopping**: Implement early stopping to prevent overfitting by monitoring validation loss.